In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from typing import Iterable, List

from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

from scipy.stats import randint, uniform

from catboost import CatBoostRegressor
import xgboost as xgb

from feature_utils import Catch22FeatureExtractor, WeatherFeaturesExtractor, DFColumnTransformer, CyclicFeaturesExtractor

In [ ]:
data = pd.read_csv('./data/merged_data.csv')

In [ ]:
def _drop_cols_fn(X, cols: Iterable[str]):
    if isinstance(X, pd.DataFrame):
        return X.drop(columns=[c for c in cols if c in X.columns], errors='ignore')
    return X

In [ ]:
num_cols = ['barcelona temp', 'seville temp', 'madrid temp', 'valencia temp', 'bilbao temp', 'seville wind_speed', 'barcelona wind_speed', 'madrid wind_speed', 'valencia wind_speed', 'bilbao wind_speed', 'seville wind_deg', 'barcelona wind_deg', 'madrid wind_deg', 'valencia wind_deg', 'bilbao wind_deg', 'seville humidity', 'barcelona humidity', 'madrid humidity', 'valencia humidity', 'bilbao humidity', 'seville pressure', 'barcelona pressure', 'madrid pressure', 'valencia pressure', 'bilbao pressure', 'seville rain_1h', 'barcelona rain_1h', 'madrid rain_1h', 'valencia rain_1h', 'bilbao rain_1h', 'day', 'hour', 'month', 'day of week']
cat_cols = ['vacation', 'seville weather_description', 'barcelona weather_description', 'madrid weather_description', 'valencia weather_description', 'bilbao weather_description']
catch22_cols = ['barcelona temp', 'seville temp', 'madrid temp', 'valencia temp', 'bilbao temp', 'seville wind_speed', 'barcelona wind_speed', 'madrid wind_speed', 'valencia wind_speed', 'bilbao wind_speed', 'seville wind_deg', 'barcelona wind_deg', 'madrid wind_deg', 'valencia wind_deg', 'bilbao wind_deg', 'seville humidity', 'barcelona humidity', 'madrid humidity', 'valencia humidity', 'bilbao humidity', 'seville pressure', 'barcelona pressure', 'madrid pressure', 'valencia pressure', 'bilbao pressure', 'seville rain_1h', 'barcelona rain_1h', 'madrid rain_1h', 'valencia rain_1h', 'bilbao rain_1h']
# NOTE: future weather features assuming forecast is available
catch22_win_past = [12, 24, 7*24]
catch22_win_future = [12, 24, 24]
cities = ['seville', 'madrid', 'barcelona', 'valencia', 'bilbao']
cols_to_drop = ['time', 'month', 'day', 'day of week']
cyclic_cols = ['month', 'hour', 'day of week']
cyclic_vals = [12, 24, 7]

feature_pipeline = Pipeline(steps=[
    ('weather_fea_extraction', WeatherFeaturesExtractor(cities=cities)),
    ('catch22_fea_extraction', Catch22FeatureExtractor(target_cols=catch22_cols, windows_past_hrs=catch22_win_past, windows_future_hrs=catch22_win_future, njobs_default=16)),
    ('cyclic_fea_extractor', CyclicFeaturesExtractor(cyclic_cols, cyclic_vals)),
    ('col_drop', FunctionTransformer(func=_drop_cols_fn, kw_args={'cols': cols_to_drop}, validate=False)),
    #('imputing', DFColumnTransformer(transformers=[
    #    ('num', SimpleImputer(strategy='mean'), num_cols),
    #    ('cat', SimpleImputer(strategy='most_frequent'), cat_cols)
    #], remainder='passthrough'))
], verbose=True)

In [ ]:
data['time'] = pd.to_datetime(data['time']).reset_index(drop=True)
data = data.sort_values('time', ascending=True)
data[cat_cols] = data[cat_cols].astype('category')

data_trn_val, data_tst = train_test_split(data, test_size=0.25, shuffle=False)
data_trn_val = data_trn_val.reset_index(drop=True)
data_tst = data_tst.reset_index(drop=True)
data_trn, data_val = train_test_split(data_trn_val, test_size=0.31, shuffle=False)
data_trn, data_val = data_trn.reset_index(drop=True), data_val.reset_index(drop=True)

target_cols = ['renewable generation percent']

X_trn, y_trn = data_trn[cat_cols + num_cols + cols_to_drop], data_trn[target_cols]
X_val, y_val = data_val[cat_cols + num_cols + cols_to_drop], data_val[target_cols]
X_tst, y_tst = data_tst[cat_cols + num_cols + cols_to_drop], data_tst[target_cols]

# crop the labels
start_crop = np.max(catch22_win_past)
end_crop = np.max(catch22_win_future)

y_trn = y_trn.iloc[start_crop:len(y_trn)-end_crop]
y_val = y_val.iloc[start_crop:len(y_val)-end_crop]
y_tst = y_tst.iloc[start_crop:len(y_tst)-end_crop]

# transform the features
X_trn = feature_pipeline.fit_transform(X_trn)
X_val = feature_pipeline.transform(X_val)
X_tst = feature_pipeline.transform(X_tst)

assert len(X_trn) == len(y_trn)
assert len(X_val) == len(y_val)
assert len(X_tst) == len(y_tst)

def joint_delete_nan_cols(dfs: List[pd.DataFrame], thresh_perc: float) -> None:
    cols_to_delete = []
    for col in dfs[0].columns:
        for df in dfs:
            if pd.isna(df[col]).mean() >= thresh_perc:
                cols_to_delete.append(col)
                break
    print(f'deleting: {len(cols_to_delete)} cols')

    for df in dfs:
        df.drop(columns=cols_to_delete, inplace=True)

joint_delete_nan_cols([X_trn, X_val, X_tst], thresh_perc=0.05)

In [ ]:
# define and fit the model
model = CatBoostRegressor(
    iterations=8000,
    random_state=42,
    od_wait=400,
    od_type='Iter',
    cat_features=cat_cols,
    l2_leaf_reg=100
)

model.fit(X=X_trn, y=y_trn, eval_set=(X_val, y_val), verbose_eval=100)

In [ ]:
y_pred = model.predict(X_tst)

print(f'MAE: {mean_absolute_error(y_tst, y_pred)}')
print(f'RMSE: {root_mean_squared_error(y_tst, y_pred)}')
print(f'R2: {r2_score(y_tst, y_pred)}')

In [ ]:
def plot_hourly_daily_weekly(pred: np.ndarray,
                             gt: np.ndarray,
                             start_time: pd.Timestamp,
                             start_hour: int = 0):
    """
    Plot hourly, daily, and weekly data by simple block averaging.

    pred, gt : 1D arrays or pandas Series of hourly data (same length)
    start_time : pandas Timestamp marking the start of the data
    start_hour : integer offset (for labeling consistency)
    """

    # --- convert to numpy if Series ---
    if hasattr(gt, "to_numpy"):
        gt = gt.to_numpy()
    if hasattr(pred, "to_numpy"):
        pred = pred.to_numpy()

    assert len(pred) == len(gt), "Prediction and ground truth must have same length"
    
    n = len(pred)
    hours = np.arange(start_hour, start_hour + n)

    # --- create datetime index for hourly data ---
    time_index = pd.date_range(start=start_time, periods=n, freq="H")

    # --- block averaging ---
    def block_avg(x, block_size):
        n_blocks = n // block_size
        return x[:n_blocks * block_size].reshape(n_blocks, block_size).mean(axis=1)

    daily_pred = block_avg(pred, 24)
    daily_gt   = block_avg(gt, 24)
    weekly_pred = block_avg(pred, 24 * 7)
    weekly_gt   = block_avg(gt, 24 * 7)

    # --- create datetime indices for daily and weekly averages ---
    daily_index  = pd.date_range(start=start_time, periods=len(daily_pred), freq="D")
    weekly_index = pd.date_range(start=start_time, periods=len(weekly_pred), freq="W")

    # --- plot ---
    fig, axs = plt.subplots(3, 1, figsize=(20, 10), sharex=False)

    # Hourly
    axs[0].plot(time_index, gt, label='GT', lw=1.2)
    axs[0].plot(time_index, pred, label='Pred', lw=1.2, alpha=0.7)
    axs[0].set_title('Hourly')
    axs[0].legend()

    # Daily
    axs[1].plot(daily_index, daily_gt, label='GT', lw=1.5)
    axs[1].plot(daily_index, daily_pred, label='Pred', lw=1.5, alpha=0.7)
    axs[1].set_title('Daily (block avg)')
    axs[1].legend()

    # Weekly
    axs[2].plot(weekly_index, weekly_gt, label='GT', lw=2)
    axs[2].plot(weekly_index, weekly_pred, label='Pred', lw=2, alpha=0.7)
    axs[2].set_title('Weekly (block avg)')
    axs[2].legend()

    # --- Format x-axis for dates ---
    for ax in axs:
        ax.grid(True)
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
        plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

    plt.tight_layout()
    plt.show()


In [ ]:
plot_hourly_daily_weekly(y_pred, y_tst, start_time=data_tst['time'].min())

In [ ]:
feature_importance = model.get_feature_importance(prettified=True)
for name, importance in zip(feature_importance['Feature Id'], feature_importance['Importances']):
    print(f"{name:<30} ---> {importance:.6f}")

In [ ]:
model.save_model('./cat_model.cbm')

In [ ]:
model.load_model('./cat_model.cbm')

y_pred = model.predict(X_tst)

print(f'MAE: {mean_absolute_error(y_tst, y_pred)}')
print(f'RMSE: {root_mean_squared_error(y_tst, y_pred)}')
print(f'R2: {r2_score(y_tst, y_pred)}')

In [ ]:
xgboost_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1, verbosity=0, tree_method='hist', enable_categorical=True)

# crossvalidation on training set
xgboost_grid_search_params = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(3, 9),
    'learning_rate': uniform(0.03, 0.3),
    'colsample_bytree': uniform(0.2, 0.8),
    #'subsample': uniform(0.1, 0.95),
    #'gamma': uniform(0.0, 5.0),
    #'min_child_weight': uniform(0.1, 10),
    #'objective': ['reg:squarederror', 'reg:tweedie', 'reg:gamma', 'reg:absoluteerror']
}

xgboost_grid_search = RandomizedSearchCV(
    estimator=xgboost_model,
    param_distributions=xgboost_grid_search_params,
    n_iter=5,
    scoring='r2',
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=4,
    pre_dispatch='6*n_jobs'
)
# fit the grid search
xgboost_grid_search.fit(X_trn, y_trn)

xgboost_model = xgboost_grid_search.best_estimator_
print("Best parameters found: ", xgboost_grid_search.best_params_)
print("Best cross-validation R2: ", xgboost_grid_search.best_score_)

# evaluate on testing set
xgboost_model.fit(X_trn, y_trn)
y_pred = xgboost_model.predict(X_tst)


print(f'MAE: {mean_absolute_error(y_tst, y_pred)}')
print(f'RMSE: {root_mean_squared_error(y_tst, y_pred)}')
print(f'R2: {r2_score(y_tst, y_pred)}')

In [ ]:
plot_hourly_daily_weekly(y_pred, y_tst, start_time=data_tst['time'].min())

In [ ]:
xgboost_model.save_model('./xgb_model.ubj')